# QE pipeline: ThinkQE и GenCRF — 6 комбинаций × 2 датасета

**Ретриверы**: `bm25`, `giga-embeddings`, `RRF(giga + bm25)` (вариант B — расширение применяется только к bm25-ветке).

**QE-методы**: `ThinkQE`, `GenCRF` — оба через **DeepSeek API**.

* **ThinkQE** (Lei et al., 2025): итеративный thinking-процесс с feedback от BM25-корпуса. На раунде *t* LLM «размышляет» с учётом топ-K документов из BM25(qₜ, C), генерирует расширение, оно идёт в следующий раунд. Финальное расширение — конкатенация всех expansion'ов через все раунды.
* **GenCRF** (Seo et al., 2024): генерируется три типа переформулировок — *contextual expansion*, *detail specific*, *aspect specific* — затем переформулировки кластеризуются для устранения избыточности, и вычисляется взвешенная агрегация (по умолчанию ScoreDW/SimDW). Реализуем **SimDW**-вариант (similarity-based dynamic weighting) — он не требует supervision-сигналов, в отличие от QERM. Финальное расширение — взвешенная конкатенация представителей кластеров.

**Датасеты**: `kaengreg/rus-scifact`, `kaengreg/rus-nfcorpus` (полные test-split).

## Алгоритм

Запросы и документы препроцессятся **одинаково**: lowercase + токенизация + лемматизация (pymorphy3) + удаление пунктуации + мягкая фильтрация стоп-слов (без вопросительных и отрицаний). Колонки `processed_text` / `processed_title` из датасетов **не используются**.

Для каждой комбинации `(retriever, qe_method, dataset)`:

1. Берём оригинальный `query.text`.
2. Применяем QE-метод → получаем расширение (генерация на сыром тексте, без предобработки).
3. `expanded = original + ' ' + expansion`.
4. `processed = preprocess(expanded)` — для bm25-ветки.
5. Поиск top-100. Для giga (как самостоятельного ретривера) подаётся `expanded` сырым (с E5-style префиксом). Для giga-ветки в RRF-B подаётся `original` сырым.

Корпус препроцессится **один раз на датасет** тем же `preprocess()` и кэшируется на диск (`cache/corpus_proc_{dataset}.json`).

## Артефакты

* **12 файлов расширений** в `expansions/{retriever}_{qe_method}_{dataset}.json` — тройки `{qid, original, expanded, processed}`.
* **12 файлов прогонов** в `runs/{retriever}_{qe_method}_{dataset}.json` — top-100 doc_id со скорами на запрос.
* **Сводная таблица метрик** в `results/qe_summary_thinkqe_gencrf.csv`.

## 0. Установка зависимостей

In [ ]:
!pip install -q datasets 'transformers==4.51.0' 'sentence-transformers==5.1.1' accelerate faiss-gpu-cu12 rank-bm25 pytrec_eval tqdm einops openai pymorphy3 scikit-learn pandas nest_asyncio

In [ ]:
from __future__ import annotations

import os
import re
import gc
import json
import math
import time
import asyncio
from collections import Counter, defaultdict
from pathlib import Path
from typing import Callable

import numpy as np
import torch
import faiss
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import pytrec_eval
import pymorphy3
from rank_bm25 import BM25Okapi

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

# --- Директории артефактов -----------------------------------------------
ROOT = Path('.')
EXP_DIR = ROOT / 'expansions'   # 24 файла расширений
RUN_DIR = ROOT / 'runs'         # 24 файла прогонов
RES_DIR = ROOT / 'results'      # сводные метрики
CACHE_DIR = ROOT / 'cache'      # эмбеддинги, BM25-индексы и т.д.
for d in (EXP_DIR, RUN_DIR, RES_DIR, CACHE_DIR):
    d.mkdir(exist_ok=True)

# --- DeepSeek API ---------------------------------------------------------
DEEPSEEK_API_KEY_PLACEHOLDER'DEEPSEEK_API_KEY', '')
DEEPSEEK_MODEL = 'deepseek-chat'
DEEPSEEK_BASE_URL = 'https://api.deepseek.com'
DEEPSEEK_MAX_CONCURRENCY = 32  # параллельных запросов к API

if not DEEPSEEK_API_KEY_PLACEHOLDER    print('WARNING: DEEPSEEK_API_KEY не задан в env — установите перед прогоном QE-методов.')

## 1. Загрузка датасетов

In [ ]:
def load_beir_like(name: str, qrels_split: str = 'test'):
    corpus_ds = load_dataset(f'kaengreg/{name}', 'corpus')['train']
    queries_ds = load_dataset(f'kaengreg/{name}', 'queries')['train']
    qrels_ds = load_dataset(f'kaengreg/{name}-qrels', 'qrels')[qrels_split]

    corpus = {}
    for row in corpus_ds:
        corpus[str(row['_id'])] = {
            'title': row.get('title') or '',
            'text': row.get('text') or '',
        }

    qrels = defaultdict(dict)
    for row in qrels_ds:
        qrels[str(row['query-id'])][str(row['corpus-id'])] = int(row['score'])

    valid_qids = set(qrels.keys())
    queries = {}
    for row in queries_ds:
        qid = str(row['_id'])
        if qid in valid_qids:
            queries[qid] = {'text': row.get('text') or ''}
    qrels = {q: r for q, r in qrels.items() if q in queries}

    print(f'{name}: corpus={len(corpus)}, queries={len(queries)}, qrels={len(qrels)}')
    return corpus, queries, dict(qrels)


DATASETS = {
    'rus-scifact': load_beir_like('rus-scifact', 'test'),
    'rus-nfcorpus': load_beir_like('rus-nfcorpus', 'test'),
}

## 2. Препроцессинг для BM25

Для русского IR удалять стоп-слова надо аккуратно: убираем местоимения / союзы / частицы / предлоги, но **сохраняем** вопросительные слова (`что`, `как`, `какой`, `где`, `когда`, `почему`, `сколько` и т.д.) и отрицания (`не`, `нет`, `ни`) — они несут смысл в IR-запросах.

Список ниже — это nltk-russian минус вопросительные/отрицания/значимые наречия.

In [ ]:
# Список «безопасных» стоп-слов: без вопросительных слов и отрицаний.
# Базируется на nltk-russian, очищен от информативных токенов.
SOFT_RU_STOPWORDS = frozenset("""
и в во на с со от до из у к о об про над под при за по для
а но или либо да же ли бы б
я ты он она оно мы вы они меня тебя его её нас вас их
мне тебе ему ей нам вам им мной тобой нами вами ими
себя себе собой
это эта этот эти то та тот те
был была было были быть есть
только лишь уже ещё тоже также
вот ведь ну ах ой эх
если иначе чтобы хотя пока
""".split())

_morph = pymorphy3.MorphAnalyzer()
_TOKEN_RE = re.compile(r'[а-яёa-z0-9]+', re.IGNORECASE)
_lemma_cache: dict[str, str] = {}


def _lemma(token: str) -> str:
    cached = _lemma_cache.get(token)
    if cached is not None:
        return cached
    cached = _morph.parse(token)[0].normal_form
    _lemma_cache[token] = cached
    return cached


def preprocess(text: str) -> list[str]:
    """lowercase → tokenize → strip punct → lemmatize → soft stopword filter."""
    text = text.lower()
    out = []
    for raw in _TOKEN_RE.findall(text):
        if len(raw) < 2:
            continue
        lemma = _lemma(raw)
        if lemma in SOFT_RU_STOPWORDS:
            continue
        out.append(lemma)
    return out


def preprocess_str(text: str) -> str:
    return ' '.join(preprocess(text))


# Smoke test
print(preprocess_str('Какие методы машинного обучения используются для анализа геномных данных?'))

## 3. Метрики (pytrec_eval)

In [ ]:
K_VALUES = [10, 100]


def evaluate_run(qrels: dict, run: dict, k_values=K_VALUES):
    metric_strs = set()
    for k in k_values:
        metric_strs.add(f'ndcg_cut.{k}')
        metric_strs.add(f'recall.{k}')
        metric_strs.add(f'map_cut.{k}')
    evaluator = pytrec_eval.RelevanceEvaluator(qrels, metric_strs)
    per_q = evaluator.evaluate(run)
    agg = defaultdict(list)
    for q, m in per_q.items():
        for k, v in m.items():
            agg[k].append(v)
    means = {k: float(np.mean(v)) for k, v in agg.items()}
    out = {}
    for k in k_values:
        out[f'NDCG@{k}'] = means.get(f'ndcg_cut_{k}', 0.0)
        out[f'Recall@{k}'] = means.get(f'recall_{k}', 0.0)
        out[f'MAP@{k}'] = means.get(f'map_cut_{k}', 0.0)
    return out

## 4. Препроцессинг корпуса + BM25-ретривер

Корпус препроцессится тем же `preprocess()`, что и запросы. Делается это **один раз на датасет** и сохраняется в `cache/corpus_proc_{dataset}.json`. После этого BM25-индекс строится поверх этих токенов.

In [ ]:
_BM25_INDEX_CACHE: dict[str, tuple] = {}


def preprocess_corpus(corpus: dict, dataset_name: str) -> dict[str, list[str]]:
    """Прогоняет каждый документ через preprocess(title + ' ' + text).
    Кэшируется на диск — на nfcorpus 3.6k док-ов это ~10–30 секунд первый раз,
    далее моментально.
    Возвращает: {doc_id: [token, ...]}
    """
    cache = CACHE_DIR / f'corpus_proc_{dataset_name}.json'
    if cache.exists():
        return json.loads(cache.read_text(encoding='utf-8'))
    print(f'  [preprocess] tokenizing+lemmatizing corpus for {dataset_name} '
          f'({len(corpus):,} docs)...')
    out: dict[str, list[str]] = {}
    for did, doc in tqdm(corpus.items(), desc=f'preprocess/{dataset_name}'):
        text = (doc['title'] + ' ' + doc['text']).strip()
        out[did] = preprocess(text)
    cache.write_text(json.dumps(out, ensure_ascii=False), encoding='utf-8')
    return out


def get_bm25_index(corpus: dict, dataset_name: str):
    if dataset_name in _BM25_INDEX_CACHE:
        return _BM25_INDEX_CACHE[dataset_name]
    proc = preprocess_corpus(corpus, dataset_name)
    doc_ids = list(proc.keys())
    print(f'  [BM25] building index for {dataset_name} ({len(doc_ids):,} docs)...')
    bm25 = BM25Okapi([proc[d] for d in doc_ids])
    _BM25_INDEX_CACHE[dataset_name] = (bm25, doc_ids)
    return bm25, doc_ids


def bm25_search(corpus: dict, dataset_name: str,
                processed_queries: dict, k: int = 100) -> dict:
    """processed_queries: {qid: list[token]} — запросы УЖЕ предобработанные."""
    bm25, doc_ids = get_bm25_index(corpus, dataset_name)
    run = {}
    for qid, q_tokens in tqdm(processed_queries.items(), desc=f'BM25 search {dataset_name}'):
        if not q_tokens:
            run[qid] = {}
            continue
        scores = bm25.get_scores(q_tokens)
        kk = min(k, len(scores))
        top_idx = np.argpartition(-scores, kk - 1)[:kk]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        run[qid] = {doc_ids[i]: float(scores[i]) for i in top_idx}
    return run

## 5. Giga-embeddings ретривер

Грузим один раз, кэшируем эмбеддинги документов на диск, потом для каждого QE-метода энкодим только запросы. По условию задачи (RRF вариант B) giga получает **оригинальный** запрос — не расширенный.

In [ ]:
GIGA_HF_ID = 'ai-sage/Giga-Embeddings-instruct'
GIGA_QUERY_PREFIX = ('Instruct: Дан вопрос, найди подходящий документ '
                    'который отвечает на вопрос\nQuestion: ')
GIGA_BATCH = 16
GIGA_MAX_LEN = 4096
GIGA_DTYPE = torch.bfloat16

_GIGA_MODEL = None
_GIGA_TOK = None


def load_giga():
    global _GIGA_MODEL, _GIGA_TOK
    if _GIGA_MODEL is not None:
        return _GIGA_MODEL, _GIGA_TOK
    print(f'Loading {GIGA_HF_ID} ...')
    _GIGA_TOK = AutoTokenizer.from_pretrained(GIGA_HF_ID, trust_remote_code=True)
    kw = dict(trust_remote_code=True, torch_dtype=GIGA_DTYPE)
    try:
        _GIGA_MODEL = AutoModel.from_pretrained(
            GIGA_HF_ID, attn_implementation='flash_attention_2', **kw
        ).to(DEVICE)
    except (ImportError, ValueError) as e:
        print(f'  flash_attention_2 unavailable ({type(e).__name__}); using eager.')
        _GIGA_MODEL = AutoModel.from_pretrained(GIGA_HF_ID, **kw).to(DEVICE)
    _GIGA_MODEL.eval()
    return _GIGA_MODEL, _GIGA_TOK


@torch.inference_mode()
def giga_encode(texts: list[str], prefix: str = '', desc: str = 'giga') -> np.ndarray:
    model, tok = load_giga()
    if prefix:
        texts = [prefix + t for t in texts]
    order = np.argsort([len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]
    out = [None] * len(texts)
    for s in tqdm(range(0, len(sorted_texts), GIGA_BATCH), desc=desc):
        batch = sorted_texts[s:s + GIGA_BATCH]
        enc = tok(batch, padding=True, truncation=True,
                  max_length=GIGA_MAX_LEN, return_tensors='pt').to(DEVICE)
        emb = model(**enc, return_embeddings=True)
        if not torch.is_tensor(emb):
            emb = emb[0] if isinstance(emb, (tuple, list)) else emb.embeddings
        emb = torch.nn.functional.normalize(emb, p=2, dim=1).float().cpu().numpy()
        for i, idx in enumerate(order[s:s + GIGA_BATCH]):
            out[idx] = emb[i]
    return np.vstack(out)


def faiss_search_gpu(doc_emb: np.ndarray, query_emb: np.ndarray, k: int = 100):
    index = faiss.IndexFlatIP(doc_emb.shape[1])
    res = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)
    index.add(doc_emb.astype(np.float32))
    return index.search(query_emb.astype(np.float32), k)


def get_giga_doc_emb(corpus: dict, dataset_name: str) -> tuple[np.ndarray, list[str]]:
    cache = CACHE_DIR / f'giga_doc_{dataset_name}.npy'
    cache_ids = CACHE_DIR / f'giga_doc_ids_{dataset_name}.json'
    if cache.exists() and cache_ids.exists():
        return np.load(cache), json.loads(cache_ids.read_text())
    doc_ids = list(corpus.keys())
    doc_texts = [(corpus[d]['title'] + ' ' + corpus[d]['text']).strip() for d in doc_ids]
    emb = giga_encode(doc_texts, prefix='', desc=f'giga/docs/{dataset_name}')
    np.save(cache, emb)
    cache_ids.write_text(json.dumps(doc_ids))
    return emb, doc_ids


def giga_search(corpus: dict, dataset_name: str,
                qid2text: dict, k: int = 100) -> dict:
    """qid2text: {qid: str} — сырые запросы (без препроцессинга, БЕЗ префикса)."""
    doc_emb, doc_ids = get_giga_doc_emb(corpus, dataset_name)
    qids = list(qid2text.keys())
    q_texts = [qid2text[q] for q in qids]
    q_emb = giga_encode(q_texts, prefix=GIGA_QUERY_PREFIX, desc=f'giga/queries/{dataset_name}')
    scores, idxs = faiss_search_gpu(doc_emb, q_emb, k=k)
    run = {}
    for i, qid in enumerate(qids):
        run[qid] = {}
        for j in range(idxs.shape[1]):
            di = idxs[i, j]
            if di < 0:
                continue
            run[qid][doc_ids[di]] = float(scores[i, j])
    return run

## 6. RRF (вариант B)

Сливаем `giga(оригинальный_запрос)` и `bm25(расширенный+предобработанный_запрос)`. Стандартный k=60.

In [ ]:
def rrf_fuse(runs: list[dict], k_rrf: int = 60, top_k: int = 100) -> dict:
    qids = set().union(*[set(r.keys()) for r in runs])
    fused = {}
    for qid in qids:
        agg = defaultdict(float)
        for r in runs:
            if qid not in r:
                continue
            ranked = sorted(r[qid].items(), key=lambda x: -x[1])
            for rank, (did, _) in enumerate(ranked, start=1):
                agg[did] += 1.0 / (k_rrf + rank)
        top = sorted(agg.items(), key=lambda x: -x[1])[:top_k]
        fused[qid] = {d: float(s) for d, s in top}
    return fused

## 7. DeepSeek API: drop-in замена для vLLM-обёрток

QE-методы (`method_thinkqe`, `method_gencrf`) написаны в стилистике `qe_methods.ipynb` и ожидают vLLM-совместимый интерфейс. Дадим его через DeepSeek API.

* `make_llm()` возвращает `(client, sp_cls)`, где `client` — наш wrapper, а `sp_cls` — фейковый класс с теми же атрибутами, что у `vllm.SamplingParams` (используется только для совместимости сигнатур).
* `vllm_generate(llm, sp_cls, prompts, ...)` и `vllm_sample_n_batch(...)` — async-под-капотом, конкурентность через семафор.

DeepSeek API совместимо с OpenAI SDK — используем `AsyncOpenAI` с `base_url`.

In [ ]:
from openai import AsyncOpenAI


class _SamplingParams:
    """Имитация vllm.SamplingParams — мы используем только поля, которые читаем сами."""
    def __init__(self, temperature=0.0, top_p=1.0, max_tokens=128, n=1):
        self.temperature = temperature
        self.top_p = top_p
        self.max_tokens = max_tokens
        self.n = n


class DeepSeekClient:
    def __init__(self, api_key: str, base_url: str = DEEPSEEK_BASE_URL,
                 model: str = DEEPSEEK_MODEL, max_concurrency: int = DEEPSEEK_MAX_CONCURRENCY):
        self.client = AsyncOpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.sem = asyncio.Semaphore(max_concurrency)

    async def _one(self, messages, sp: _SamplingParams, n_override: int | None = None):
        # n>1 эмулируем серией параллельных запросов с одним и тем же promptом
        # (DeepSeek не всегда корректно поддерживает n>1).
        n = n_override if n_override is not None else sp.n
        if n == 1:
            async with self.sem:
                for attempt in range(4):
                    try:
                        resp = await self.client.chat.completions.create(
                            model=self.model,
                            messages=messages,
                            temperature=sp.temperature,
                            top_p=sp.top_p,
                            max_tokens=sp.max_tokens,
                        )
                        return [resp.choices[0].message.content or '']
                    except Exception as e:
                        if attempt == 3:
                            print(f'  [deepseek] error after retries: {e}')
                            return ['']
                        await asyncio.sleep(2 ** attempt)
        else:
            tasks = [self._one(messages, sp, n_override=1) for _ in range(n)]
            chunks = await asyncio.gather(*tasks)
            return [c[0] for c in chunks]

    async def _batch(self, all_messages, sp: _SamplingParams):
        """Возвращает list[list[str]] длиной len(all_messages); каждый внутренний список длины sp.n."""
        tasks = [self._one(m, sp) for m in all_messages]
        return await asyncio.gather(*tasks)

    def get_tokenizer(self):
        # Совместимость с vllm-style render_chat — возвращаем None, render_chat ниже
        # обработает это и просто отдаст messages в OpenAI формат.
        return None


def make_llm():
    if not DEEPSEEK_API_KEY_PLACEHOLDER        raise RuntimeError('Set DEEPSEEK_API_KEY env var before using QE methods.')
    return DeepSeekClient(DEEPSEEK_API_KEY), _SamplingParams

In [ ]:
# --- Drop-in замена render_chat / vllm_generate / vllm_sample_n_batch ---

def _to_messages(p):
    """Принимает либо list[dict] (chat), либо str (одно user-сообщение)."""
    if isinstance(p, list):
        return p
    return [{'role': 'user', 'content': p}]


def _run_async(coro):
    """Запускает asyncio coroutine из sync-кода (включая Jupyter)."""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = None
    if loop and loop.is_running():
        # Jupyter: используем nest_asyncio при необходимости
        import nest_asyncio
        nest_asyncio.apply()
        return loop.run_until_complete(coro)
    return asyncio.run(coro)


def render_chat(llm, messages_or_str):
    """Здесь это no-op — DeepSeek принимает messages в OpenAI формате напрямую."""
    return _to_messages(messages_or_str)


def vllm_generate(llm, sp_cls, prompts, max_new_tokens=180,
                  temperature=0.0, top_p=1.0):
    """1 sample на промпт. Возвращает list[str]."""
    if not prompts:
        return []
    sp = sp_cls(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, n=1)
    msgs = [_to_messages(p) for p in prompts]
    results = _run_async(llm._batch(msgs, sp))
    return [r[0].strip() for r in results]


def vllm_sample_n_batch(llm, sp_cls, prompts, n, max_new_tokens=64,
                        temperature=1.0, top_p=1.0):
    """n samples на промпт. Возвращает list[list[str]]."""
    if not prompts:
        return []
    sp = sp_cls(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, n=n)
    msgs = [_to_messages(p) for p in prompts]
    results = _run_async(llm._batch(msgs, sp))
    return [[s.strip() for s in r] for r in results]

## 8. QE-методы: ThinkQE и GenCRF

Реализованы согласно соответствующим статьям, с минимальными адаптациями: используем DeepSeek API через те же `vllm_generate` / `vllm_sample_n_batch` обёртки, что и в основном пайплайне.

In [ ]:
# === ThinkQE (Lei, Shen, Yates, EMNLP 2025) ============================
# Итеративный процесс: thinking-based expansion + corpus-interaction feedback.
#
# Алгоритм:
#   q_0 = original query
#   for t in 1..T:
#       R_t = BM25(q_{t-1}, C).top_k
#       expansion_t = LLM_thinking(q_0, R_t, prev_expansions)
#       q_t = q_0 + ' ' + ' '.join(expansion_1..expansion_t)
#   return concat(expansion_1..expansion_T)
#
# В оригинале T=3, top_k=10. Для русского научного домена тоже используем эти значения.

THINKQE_ROUNDS = 3
THINKQE_PRF_DEPTH = 10
THINKQE_DOC_TRUNCATE = 800   # символов на документ в контексте

THINKQE_SYSTEM = (
    'Ты — ассистент, который помогает расширять поисковые запросы для научного поиска. '
    'Твоя задача — найти разнообразные термины, синонимы, связанные понятия и аспекты, '
    'которые помогут найти больше релевантных документов.'
)

THINKQE_FIRST_USER = (
    'ЗАПРОС: {QUERY}\n\n'
    'Ниже приведены документы, которые поисковая система нашла по этому запросу.\n\n'
    '{DOCS}\n\n'
    'Подумай рассуждая (think step by step):\n'
    '1. Какие подтемы и аспекты имеет данный запрос?\n'
    '2. Какие термины, синонимы и связанные понятия не упомянуты в запросе, но важны?\n'
    '3. Какие из найденных документов покрывают эти аспекты, а какие нет?\n\n'
    'Затем напиши финальное расширение запроса — список ключевых терминов, понятий '
    'и фраз (через запятую или построчно), которые расширят семантический охват.\n\n'
    'Формат ответа:\n'
    'Размышление: <твой ход рассуждения>\n'
    'Расширение: <список терминов и фраз>'
)

THINKQE_NEXT_USER = (
    'ЗАПРОС: {QUERY}\n\n'
    'Предыдущие расширения, которые ты уже сгенерировал:\n{PREV_EXP}\n\n'
    'Новые документы из поисковой системы (после применения предыдущих расширений):\n\n'
    '{DOCS}\n\n'
    'Подумай:\n'
    '1. Какие аспекты запроса всё ещё недостаточно покрыты?\n'
    '2. Какие новые термины и понятия можно добавить, чтобы охватить недостающие грани?\n'
    '3. Не повторяй то, что уже есть в предыдущих расширениях.\n\n'
    'Формат ответа:\n'
    'Размышление: <твой ход рассуждения>\n'
    'Расширение: <новые термины и фразы, которых ещё не было>'
)

_THINKQE_EXP_RE = re.compile(r'Расширение\s*:\s*(.+)$', re.DOTALL | re.IGNORECASE)


def _thinkqe_parse(text: str) -> str:
    """Достаём только часть после 'Расширение:' (отбрасываем размышление)."""
    if not text:
        return ''
    m = _THINKQE_EXP_RE.search(text)
    if m:
        return m.group(1).strip()
    # Fallback — если LLM не следует формату, берём всё
    return text.strip()


def _thinkqe_format_docs(doc_ids, id2text, truncate):
    parts = []
    for i, did in enumerate(doc_ids, start=1):
        txt = (id2text.get(did, '') or '')[:truncate]
        parts.append(f'[Документ {i}]\n{txt}')
    return '\n\n'.join(parts)


def method_thinkqe(qtexts, *,
                   bm25_retriever_fn,  # callable(qtexts: list[str], k: int) -> {qi: [doc_id]}
                   id2text,            # {doc_id: full text}
                   llm, sp_cls,
                   rounds: int = THINKQE_ROUNDS,
                   prf_depth: int = THINKQE_PRF_DEPTH,
                   doc_truncate: int = THINKQE_DOC_TRUNCATE,
                   max_new_tokens: int = 512):
    """ThinkQE: T раундов thinking + BM25 PRF.
    Возвращает list[str] — конкатенация расширений по раундам."""
    if not qtexts:
        return []
    print(f'  [ThinkQE] {len(qtexts):,} queries, {rounds} rounds, top-{prf_depth} ...')

    n = len(qtexts)
    # accumulated expansions per query — list per qi of round-by-round expansions
    accum: list[list[str]] = [[] for _ in range(n)]

    for r in range(1, rounds + 1):
        # 1) Build current query texts for retrieval = q_0 + ' ' + accumulated expansions
        cur_qtexts = []
        for qi in range(n):
            if accum[qi]:
                cur_qtexts.append(qtexts[qi] + ' ' + ' '.join(accum[qi]))
            else:
                cur_qtexts.append(qtexts[qi])

        # 2) BM25 retrieval to get top-k docs for each query
        print(f'  [ThinkQE round {r}] retrieving top-{prf_depth} docs...')
        top_per_q = bm25_retriever_fn(cur_qtexts, prf_depth)

        # 3) Build prompts
        prompts = []
        for qi in range(n):
            docs_str = _thinkqe_format_docs(
                top_per_q.get(qi, []), id2text, doc_truncate
            )
            if r == 1:
                user = THINKQE_FIRST_USER.replace('{QUERY}', qtexts[qi]).replace('{DOCS}', docs_str)
            else:
                prev = '\n'.join(f'- {e}' for e in accum[qi])
                user = (THINKQE_NEXT_USER.replace('{QUERY}', qtexts[qi])
                                          .replace('{PREV_EXP}', prev)
                                          .replace('{DOCS}', docs_str))
            prompts.append([
                {'role': 'system', 'content': THINKQE_SYSTEM},
                {'role': 'user',   'content': user},
            ])

        # 4) LLM generation
        print(f'  [ThinkQE round {r}] generating expansions...')
        outs = vllm_generate(llm, sp_cls, prompts,
                             max_new_tokens=max_new_tokens, temperature=0.7)
        for qi, raw in enumerate(outs):
            exp = _thinkqe_parse(raw)
            if exp:
                accum[qi].append(exp)

    # 5) Final expansion = concatenation of all rounds
    return [' '.join(exps) for exps in accum]

In [ ]:
# === GenCRF (Seo et al., 2024) =========================================
# Generative Clustering and Reformulation Framework.
#
# Алгоритм:
#   1. Для каждого запроса генерируем 3 типа переформулировок (по N штук каждого типа):
#      - contextual expansion: дописать контекст
#      - detail specific: добавить технические/доменные детали
#      - aspect specific: переформулировать с фокусом на разные аспекты
#   2. Все полученные ~3*N переформулировок эмбеддим, кластеризуем (например, K=3..5).
#   3. Для каждого кластера выбираем представителя (ближайшего к центроиду).
#   4. Веса представителей: SimDW — обратно-пропорционально средней попарной similarity
#      внутри кластера (разнообразные кластеры получают больший вес).
#   5. Финальное расширение — взвешенная конкатенация представителей: каждый представитель
#      повторяется num_repeats раз пропорционально своему весу.
#
# QERM (RL feedback module) опускаем — он требует обучения отдельной reward-модели и
# не относится к zero-shot test-time стадии. Используем SimDW веса напрямую.

GENCRF_N_PER_PROMPT = 3       # сколько переформулировок на каждый из 3 типов промптов
GENCRF_TEMP = 0.9
GENCRF_MAX_TOK = 64
GENCRF_K_RANGE = (2, 3, 4, 5) # пробуем эти K при кластеризации, выбираем по силуэту
GENCRF_REPEAT_SCALE = 5       # макс. повторов представителя в финальной строке

GENCRF_PROMPT_CONTEXTUAL = (
    'Перепиши данный поисковый запрос, добавив контекст и общие связанные понятия, '
    'которые помогут лучше понять, что ищет пользователь. Ответь только переписанным '
    'запросом, без пояснений.\n\n'
    'Запрос: {QUERY}'
)
GENCRF_PROMPT_DETAIL = (
    'Перепиши данный поисковый запрос, добавив конкретные технические термины, '
    'специфические детали и доменную лексику. Ответь только переписанным запросом, '
    'без пояснений.\n\n'
    'Запрос: {QUERY}'
)
GENCRF_PROMPT_ASPECT = (
    'Перепиши данный поисковый запрос, делая акцент на одном из его аспектов или граней. '
    'Каждый раз можешь выбирать новый аспект. Ответь только переписанным запросом, '
    'без пояснений.\n\n'
    'Запрос: {QUERY}'
)
GENCRF_PROMPTS = [
    ('contextual', GENCRF_PROMPT_CONTEXTUAL),
    ('detail',     GENCRF_PROMPT_DETAIL),
    ('aspect',     GENCRF_PROMPT_ASPECT),
]


def _gencrf_cluster(emb: np.ndarray, k_range=GENCRF_K_RANGE):
    """KMeans с выбором K по silhouette. Fallback к K=1 если все падают."""
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    n = emb.shape[0]
    best, best_score = None, -1.0
    for K in k_range:
        if K >= n:
            break
        try:
            km = KMeans(n_clusters=K, n_init=5, random_state=0)
            labels = km.fit_predict(emb)
            if len(set(labels)) < 2:
                continue
            s = silhouette_score(emb, labels, metric='cosine')
            if s > best_score:
                best_score, best = s, (km, labels)
        except Exception:
            continue
    if best is None:
        # Fallback: один кластер
        km = KMeans(n_clusters=1, n_init=1, random_state=0)
        labels = km.fit_predict(emb) if n > 0 else np.array([], dtype=int)
        return km, labels
    return best


def _gencrf_simdw_weights(emb: np.ndarray, labels: np.ndarray, n_clusters: int):
    """SimDW (similarity-based dynamic weighting):
    вес кластера = 1 - mean_intra_cluster_similarity.
    Кластеры с более разнообразным содержимым получают больший вес.
    Веса нормируются так, чтобы суммироваться в 1.
    """
    weights = np.zeros(n_clusters, dtype=np.float32)
    for c in range(n_clusters):
        idx = np.where(labels == c)[0]
        if len(idx) <= 1:
            weights[c] = 1.0
            continue
        sub = emb[idx]
        sims = sub @ sub.T
        # Среднее off-diagonal
        n_sub = len(idx)
        mean_sim = (sims.sum() - np.trace(sims)) / max(1, n_sub * (n_sub - 1))
        weights[c] = max(0.0, 1.0 - float(mean_sim))
    if weights.sum() <= 1e-8:
        weights = np.ones(n_clusters, dtype=np.float32)
    weights = weights / weights.sum()
    return weights


def method_gencrf(qtexts, *, llm, sp_cls, encode_fn,
                  n_per_prompt: int = GENCRF_N_PER_PROMPT,
                  repeat_scale: int = GENCRF_REPEAT_SCALE):
    """GenCRF: 3 типа промптов × N samples → cluster → SimDW-weighted concat."""
    if not qtexts:
        return []
    print(f'  [GenCRF] generating reformulations for {len(qtexts):,} queries ...')

    n = len(qtexts)
    # samples_per_q[qi] = list[str] — все переформулировки для запроса qi
    samples_per_q: list[list[str]] = [[] for _ in range(n)]

    for ptype, ptpl in GENCRF_PROMPTS:
        print(f'  [GenCRF] prompt type: {ptype}')
        prompts = [
            [{'role': 'user', 'content': ptpl.replace('{QUERY}', q)}]
            for q in qtexts
        ]
        # n_per_prompt сэмплов на каждый промпт
        outs = vllm_sample_n_batch(
            llm, sp_cls, prompts, n=n_per_prompt,
            max_new_tokens=GENCRF_MAX_TOK,
            temperature=GENCRF_TEMP, top_p=1.0,
        )
        for qi, samples in enumerate(outs):
            for s in samples:
                s = s.strip().strip('"\'').strip()
                if s and s.lower() != qtexts[qi].lower():
                    samples_per_q[qi].append(s)

    # Кластеризация и взвешенная агрегация
    print('  [GenCRF] clustering + SimDW aggregation ...')
    out: list[str] = []
    for qi in tqdm(range(n), desc='  GenCRF-cluster'):
        samples = samples_per_q[qi]
        if len(samples) < 2:
            out.append(' '.join(samples))
            continue
        emb = encode_fn(samples).astype(np.float32)
        # L2-normalize for cosine sim
        emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
        km, labels = _gencrf_cluster(emb)
        n_clusters = int(labels.max()) + 1 if len(labels) else 0
        if n_clusters <= 1:
            out.append(' '.join(samples))
            continue
        # Представитель = sample, ближайший к центроиду
        centers = km.cluster_centers_
        centers = centers / (np.linalg.norm(centers, axis=1, keepdims=True) + 1e-9)
        reps = []
        for c in range(n_clusters):
            idx = np.where(labels == c)[0]
            if len(idx) == 0:
                continue
            sub_emb = emb[idx]
            sims = sub_emb @ centers[c]
            rep_idx = idx[int(np.argmax(sims))]
            reps.append((c, samples[rep_idx]))
        # SimDW веса
        w = _gencrf_simdw_weights(emb, labels, n_clusters)
        # Финальная строка: представители повторяются пропорционально весу
        max_w = w.max() or 1.0
        parts = []
        for c, rep_text in reps:
            n_rep = max(1, int(round((w[c] / max_w) * repeat_scale)))
            parts.extend([rep_text] * n_rep)
        out.append(' '.join(parts))
    return out

## 9. Генерация расширений

Расширения для каждого `(qe_method, dataset)` генерируем **один раз**, кэшируем как `expansions/__raw_{method}_{dataset}.json` (тройка `original → expanded`). Затем делаем 12 файлов (по комбинациям) — они физически идентичны для bm25 и rrf, но это сделано для прозрачности артефактов.

Для **giga** в варианте RRF-B расширение по-прежнему генерируется (для вывода в файл и для bm25-ветки), но в search-функции giga получит **оригинальный** запрос. Для трека «giga + QE» (отдельный ретривер из 6 комбинаций) giga получает расширенный запрос как обычно.

In [ ]:
QE_METHODS = ['thinkqe', 'gencrf']
RETRIEVERS = ['bm25', 'giga', 'rrf']  # rrf == RRF(giga + bm25), вариант B


def make_bm25_retriever(corpus, dataset_name):
    """Для ThinkQE — нужен retriever_fn(qtexts, k) -> {qi: [doc_id]}.
    Запросы препроцессятся тем же preprocess() (как везде в pipeline)."""
    bm25, doc_ids = get_bm25_index(corpus, dataset_name)

    def fn(qtexts, k):
        out = {}
        for qi, q in enumerate(qtexts):
            tokens = preprocess(q)
            if not tokens:
                out[qi] = []
                continue
            scores = bm25.get_scores(tokens)
            kk = min(k, len(scores))
            top_idx = np.argpartition(-scores, kk - 1)[:kk]
            top_idx = top_idx[np.argsort(-scores[top_idx])]
            out[qi] = [doc_ids[i] for i in top_idx]
        return out
    return fn


def make_gencrf_encoder():
    """Для GenCRF — нужен encode_fn(list[str]) -> np.ndarray.
    Используем giga-embeddings (ту же, что в основном dense-ретривере)."""
    def fn(texts):
        return giga_encode(texts, prefix=GIGA_QUERY_PREFIX, desc='gencrf/encode')
    return fn


def generate_expansions_for(method: str, corpus: dict, queries: dict,
                            dataset_name: str, llm, sp_cls) -> dict:
    """Возвращает {qid: expanded_text}. Кэшируется per (method, dataset)."""
    cache = EXP_DIR / f'__raw_{method}_{dataset_name}.json'
    if cache.exists():
        return json.loads(cache.read_text(encoding='utf-8'))

    qids = list(queries.keys())
    qtexts = [queries[q]['text'] for q in qids]

    print(f'\n=== {method} on {dataset_name} ({len(qids)} queries) ===')
    if method == 'thinkqe':
        bm25_fn = make_bm25_retriever(corpus, dataset_name)
        id2text = {d: (corpus[d]['title'] + ' ' + corpus[d]['text']).strip()
                   for d in corpus}
        expansions = method_thinkqe(
            qtexts,
            bm25_retriever_fn=bm25_fn,
            id2text=id2text,
            llm=llm, sp_cls=sp_cls,
            rounds=THINKQE_ROUNDS,
            prf_depth=THINKQE_PRF_DEPTH,
        )
    elif method == 'gencrf':
        expansions = method_gencrf(
            qtexts, llm=llm, sp_cls=sp_cls,
            encode_fn=make_gencrf_encoder(),
        )
    else:
        raise ValueError(method)

    out = dict(zip(qids, expansions))
    cache.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
    return out

In [ ]:
# Генерируем 2 × 2 = 4 набора расширений
# (это шаг с DeepSeek-вызовами — самый дорогой по времени и API).

llm, sp_cls = make_llm()

raw_expansions = {}  # {(method, dataset): {qid: expanded}}
for ds_name, (corpus, queries, _) in DATASETS.items():
    for method in QE_METHODS:
        t0 = time.time()
        raw_expansions[(method, ds_name)] = generate_expansions_for(
            method, corpus, queries, ds_name, llm, sp_cls
        )
        print(f'  done in {time.time() - t0:.1f}s')

## 10. Сохранение 12 файлов расширений

Для каждой комбинации `(retriever, qe_method, dataset)` сохраняем тройки `{qid, original, expanded, processed}`.

* Для **bm25** и для **giga** (как самостоятельного ретривера) поле `expanded` = `original + ' ' + expansion`, а `processed` = preprocess(expanded).
* Для **rrf** (вариант B): `expanded` = такой же как у bm25 (т.к. расширение применяется к bm25-ветке), `processed` тоже как у bm25; giga в RRF-B получает оригинальный запрос. В файл записываем то, что реально пойдёт в bm25-ветку RRF.

Итого 3 × 2 × 2 = **12 файлов**.

In [ ]:
def build_expansion_records(retriever: str, method: str, dataset_name: str,
                            queries: dict, raw_exp: dict) -> list[dict]:
    """Строит список троек (qid, original, expanded, processed).

    `expanded` — это то, что пойдёт в ретривер ДО предобработки.
    Для bm25 и giga: original + ' ' + expansion.
    Для rrf (вариант B): то же самое — это то, что увидит bm25-ветка;
        giga-ветка получит original независимо.
    `processed` — preprocess(expanded).
    """
    records = []
    for qid in queries:
        original = queries[qid]['text']
        exp_text = raw_exp.get(qid, '')
        expanded = (original + ' ' + exp_text).strip() if exp_text else original
        processed = preprocess_str(expanded)
        records.append({
            'qid': qid,
            'original': original,
            'expanded': expanded,
            'processed': processed,
        })
    return records


def expansion_path(retriever: str, method: str, dataset_name: str) -> Path:
    return EXP_DIR / f'{retriever}_{method}_{dataset_name}.json'


def run_path(retriever: str, method: str, dataset_name: str) -> Path:
    return RUN_DIR / f'{retriever}_{method}_{dataset_name}.json'


for ds_name, (corpus, queries, _) in DATASETS.items():
    for method in QE_METHODS:
        raw = raw_expansions[(method, ds_name)]
        for retriever in RETRIEVERS:
            recs = build_expansion_records(retriever, method, ds_name, queries, raw)
            path = expansion_path(retriever, method, ds_name)
            path.write_text(json.dumps(recs, ensure_ascii=False, indent=2),
                            encoding='utf-8')

print('Saved expansion files:', len(list(EXP_DIR.glob('[!_]*.json'))))

## 11. Прогон 6 комбинаций × 2 датасета = 12 поисков

Сохраняем top-100 для каждого запроса в `runs/{retriever}_{method}_{dataset}.json`.

In [ ]:
all_metrics = []  # сводная таблица

for ds_name, (corpus, queries, qrels) in DATASETS.items():
    print(f'\n{"=" * 70}\nDataset: {ds_name}\n{"=" * 70}')

    # Кэшируем раз: эмбеддинги док-ов для giga + BM25-индекс
    get_bm25_index(corpus, ds_name)
    _ = get_giga_doc_emb(corpus, ds_name)

    # Заранее строим qid->original для giga (чтобы повторно не считать в RRF-B)
    qid2orig = {q: queries[q]['text'] for q in queries}
    giga_orig_run_path = CACHE_DIR / f'giga_orig_run_{ds_name}.json'
    if giga_orig_run_path.exists():
        giga_orig_run = json.loads(giga_orig_run_path.read_text())
    else:
        giga_orig_run = giga_search(corpus, ds_name, qid2orig, k=100)
        giga_orig_run_path.write_text(json.dumps(giga_orig_run))

    for method in QE_METHODS:
        # Загружаем сохранённые расширения (общие для всех 3 ретриверов)
        recs = json.loads(
            expansion_path('bm25', method, ds_name).read_text(encoding='utf-8')
        )
        # qid -> processed tokens (для bm25)
        qid2proc_tok = {r['qid']: r['processed'].split() for r in recs}
        # qid -> expanded raw (для giga в самостоятельном треке)
        qid2expanded = {r['qid']: r['expanded'] for r in recs}

        # ===== Ретривер: BM25 (на расширенном+предобработанном запросе) =====
        bm25_run = bm25_search(corpus, ds_name, qid2proc_tok, k=100)
        run_path('bm25', method, ds_name).write_text(
            json.dumps(bm25_run, ensure_ascii=False))
        m = evaluate_run(qrels, bm25_run)
        all_metrics.append({
            'dataset': ds_name, 'retriever': 'bm25', 'qe': method, **m
        })
        print(f'  bm25 + {method:13s}  NDCG@10={m["NDCG@10"]:.4f}  '
              f'R@100={m["Recall@100"]:.4f}')

        # ===== Ретривер: giga (на расширенном запросе как сыром тексте) =====
        giga_run = giga_search(corpus, ds_name, qid2expanded, k=100)
        run_path('giga', method, ds_name).write_text(
            json.dumps(giga_run, ensure_ascii=False))
        m = evaluate_run(qrels, giga_run)
        all_metrics.append({
            'dataset': ds_name, 'retriever': 'giga', 'qe': method, **m
        })
        print(f'  giga + {method:13s}  NDCG@10={m["NDCG@10"]:.4f}  '
              f'R@100={m["Recall@100"]:.4f}')

        # ===== Ретривер: RRF(giga[orig] + bm25[expanded]) =====
        rrf_run = rrf_fuse([giga_orig_run, bm25_run], k_rrf=60, top_k=100)
        run_path('rrf', method, ds_name).write_text(
            json.dumps(rrf_run, ensure_ascii=False))
        m = evaluate_run(qrels, rrf_run)
        all_metrics.append({
            'dataset': ds_name, 'retriever': 'rrf', 'qe': method, **m
        })
        print(f'  rrf  + {method:13s}  NDCG@10={m["NDCG@10"]:.4f}  '
              f'R@100={m["Recall@100"]:.4f}')

    # Освобождаем GPU между датасетами (на случай большого корпуса nfcorpus)
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nSaved run files: {len(list(RUN_DIR.glob("*.json")))}')

## 12. Сводная таблица

In [ ]:
import pandas as pd

df = pd.DataFrame(all_metrics)
metric_cols = ['NDCG@10', 'NDCG@100', 'Recall@10', 'Recall@100', 'MAP@10', 'MAP@100']
for col in metric_cols:
    df[col] = df[col].round(4)
df = df[['dataset', 'retriever', 'qe'] + metric_cols]
df.sort_values(['dataset', 'retriever', 'qe'], inplace=True)
df.to_csv(RES_DIR / 'qe_summary_thinkqe_gencrf.csv', index=False)
df

In [ ]:
# Удобный wide-формат: ретриверы как строки, qe-методы как колонки, метрика — NDCG@10
wide = df.pivot_table(
    index=['dataset', 'retriever'], columns='qe', values='NDCG@10'
).round(4)
wide